In [6]:
import pandas as pd
import sqlite3

print("--- BƯỚC 1: KHỞI TẠO DỮ LIỆU ---")
# Nạp tập dữ liệu gốc
df_raw = pd.read_csv('train.csv')

# Khởi tạo SQL Database ảo trên RAM
conn = sqlite3.connect(':memory:')
df_raw.to_sql('HoSoTinDung', conn, index=False)

print("=> Database 'HoSoTinDung' đã sẵn sàng.")

--- BƯỚC 1: KHỞI TẠO DỮ LIỆU ---
=> Database 'HoSoTinDung' đã sẵn sàng.


In [7]:
print("--- BƯỚC 2: SQL KIỂM CHỨNG TOÀN BỘ BIẾN PHÂN LOẠI ---")

query_categorical = """
SELECT '1. Giới tính (Gender)' AS Bien_So, Gender AS Gia_Tri, COUNT(*) AS Tong_Ho_So, ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) AS Ty_Le_Duyet_Phan_Tram FROM HoSoTinDung WHERE Gender IS NOT NULL GROUP BY Gender
UNION ALL
SELECT '2. Hôn nhân (Married)', Married, COUNT(*), ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) FROM HoSoTinDung WHERE Married IS NOT NULL GROUP BY Married
UNION ALL
SELECT '3. Người phụ thuộc (Dependents)', CAST(Dependents AS TEXT), COUNT(*), ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) FROM HoSoTinDung WHERE Dependents IS NOT NULL GROUP BY Dependents
UNION ALL
SELECT '4. Học vấn (Education)', Education, COUNT(*), ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) FROM HoSoTinDung WHERE Education IS NOT NULL GROUP BY Education
UNION ALL
SELECT '5. Tự doanh (Self_Employed)', Self_Employed, COUNT(*), ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) FROM HoSoTinDung WHERE Self_Employed IS NOT NULL GROUP BY Self_Employed
UNION ALL
SELECT '6. Lịch sử tín dụng (Credit_History)', CAST(Credit_History AS TEXT), COUNT(*), ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) FROM HoSoTinDung WHERE Credit_History IS NOT NULL GROUP BY Credit_History
UNION ALL
SELECT '7. Khu vực (Property_Area)', Property_Area, COUNT(*), ROUND(AVG(CASE WHEN Loan_Status = 'Y' THEN 1.0 ELSE 0.0 END)*100, 2) FROM HoSoTinDung WHERE Property_Area IS NOT NULL GROUP BY Property_Area;
"""

df_categorical = pd.read_sql_query(query_categorical, conn)
# Format lại để in ra bảng đẹp, căn chỉnh lề rõ ràng như ảnh yêu cầu
print(df_categorical.to_string(index=False))

print("\n=> KẾT LUẬN: Chỉ có 'Lịch sử tín dụng' và 'Khu vực' làm tỷ lệ duyệt dao động mạnh. Các cột nhân khẩu học (Giới tính, Hôn nhân...) chênh lệch rất nhỏ, không có sức mạnh phân loại rủi ro -> QUYẾT ĐỊNH XÓA!")

--- BƯỚC 2: SQL KIỂM CHỨNG TOÀN BỘ BIẾN PHÂN LOẠI ---
                             Bien_So      Gia_Tri  Tong_Ho_So  Ty_Le_Duyet_Phan_Tram
               1. Giới tính (Gender)       Female         112                  66.96
               1. Giới tính (Gender)         Male         489                  69.33
               2. Hôn nhân (Married)           No         213                  62.91
               2. Hôn nhân (Married)          Yes         398                  71.61
     3. Người phụ thuộc (Dependents)            0         345                  68.99
     3. Người phụ thuộc (Dependents)            1         102                  64.71
     3. Người phụ thuộc (Dependents)            2         101                  75.25
     3. Người phụ thuộc (Dependents)           3+          51                  64.71
              4. Học vấn (Education)     Graduate         480                  70.83
              4. Học vấn (Education) Not Graduate         134                  61.19
         5.

In [8]:
print("--- BƯỚC 3: SQL KIỂM CHỨNG CÁC BIẾN SỐ LƯỢNG (TIỀN TỆ) ---")

query_numerical = """
SELECT 
    Loan_Status AS Ket_Qua_Duyet,
    COUNT(*) AS Tong_Ho_So,
    ROUND(AVG(ApplicantIncome), 0) AS TB_ThuNhap_CaNhan,
    ROUND(AVG(CoapplicantIncome), 0) AS TB_ThuNhap_BaoLanh,
    ROUND(AVG(LoanAmount), 0) AS TB_KhoanVay,
    ROUND(AVG(Loan_Amount_Term), 0) AS TB_KyHan
FROM HoSoTinDung
GROUP BY Loan_Status;
"""
print(pd.read_sql_query(query_numerical, conn).to_string(index=False))

print("\n=> KẾT LUẬN: Thu nhập trung bình của nhóm bị từ chối (N) thậm chí ngang bằng hoặc nhỉnh hơn nhóm được duyệt (Y). Dữ liệu tiền tệ phân bổ lệch (outliers). Cần dùng Median để lấp khuyết thiếu cho khoản vay!")

--- BƯỚC 3: SQL KIỂM CHỨNG CÁC BIẾN SỐ LƯỢNG (TIỀN TỆ) ---
Ket_Qua_Duyet  Tong_Ho_So  TB_ThuNhap_CaNhan  TB_ThuNhap_BaoLanh  TB_KhoanVay  TB_KyHan
            N         192             5446.0              1878.0        151.0     344.0
            Y         422             5384.0              1505.0        144.0     341.0

=> KẾT LUẬN: Thu nhập trung bình của nhóm bị từ chối (N) thậm chí ngang bằng hoặc nhỉnh hơn nhóm được duyệt (Y). Dữ liệu tiền tệ phân bổ lệch (outliers). Cần dùng Median để lấp khuyết thiếu cho khoản vay!


In [9]:
print("--- BƯỚC 4: SQL QUÉT DỮ LIỆU KHUYẾT (MISSING VALUES) ---")

query_missing = """
SELECT 
    SUM(CASE WHEN Gender IS NULL THEN 1 ELSE 0 END) AS Miss_Gender,
    SUM(CASE WHEN Married IS NULL THEN 1 ELSE 0 END) AS Miss_Married,
    SUM(CASE WHEN Dependents IS NULL THEN 1 ELSE 0 END) AS Miss_Dependents,
    SUM(CASE WHEN Self_Employed IS NULL THEN 1 ELSE 0 END) AS Miss_SelfEmployed,
    SUM(CASE WHEN LoanAmount IS NULL THEN 1 ELSE 0 END) AS Miss_LoanAmount,
    SUM(CASE WHEN Loan_Amount_Term IS NULL THEN 1 ELSE 0 END) AS Miss_Term,
    SUM(CASE WHEN Credit_History IS NULL THEN 1 ELSE 0 END) AS Miss_CreditHistory
FROM HoSoTinDung;
"""
# Chuyển bảng dọc ra ngang (transpose) để dễ đọc
df_missing = pd.read_sql_query(query_missing, conn).T
df_missing.columns = ['So_Luong_Loi']
print(df_missing)

--- BƯỚC 4: SQL QUÉT DỮ LIỆU KHUYẾT (MISSING VALUES) ---
                    So_Luong_Loi
Miss_Gender                   13
Miss_Married                   3
Miss_Dependents               15
Miss_SelfEmployed             32
Miss_LoanAmount               22
Miss_Term                     14
Miss_CreditHistory            50


In [10]:
print("="*50)
print("BƯỚC 5: PYTHON THỰC THI LÀM SẠCH VÀ SỐ HÓA")
print("="*50)

df_clean = df_raw.copy()

# 1. XÓA CỘT NHIỄU THEO SQL (Đã bỏ 'Property_Area' ra khỏi danh sách xóa)
# Các biến nhân khẩu học này không có sức mạnh phân loại và dễ gây thiên kiến
cols_to_drop = ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed']
df_clean = df_clean.drop(columns=cols_to_drop)

# 2. XỬ LÝ KHUYẾT THIẾU (IMPUTATION)
# Lấp LoanAmount bằng Trung vị (Tránh Outlier)
df_clean['LoanAmount'] = df_clean['LoanAmount'].fillna(df_clean['LoanAmount'].median())

# Lấp Credit_History và Loan_Amount_Term bằng Số đông
df_clean['Credit_History'] = df_clean['Credit_History'].fillna(df_clean['Credit_History'].mode()[0])
df_clean['Loan_Amount_Term'] = df_clean['Loan_Amount_Term'].fillna(df_clean['Loan_Amount_Term'].mode()[0])


# 3. SỐ HÓA DỮ LIỆU (ENCODING)
# Bước bắt buộc để các thuật toán như KNN ở Phần sau có thể tính toán khoảng cách
# - Đổi nhãn mục tiêu (Target)
df_clean['Loan_Status'] = df_clean['Loan_Status'].map({'Y': 1, 'N': 0})
df_clean = df_clean.dropna(subset=['Loan_Status'])

# - Đổi chữ thành số cho cột Khu vực (Rural=0, Urban=1, Semiurban=2)
df_clean['Property_Area'] = df_clean['Property_Area'].map({'Rural': 0, 'Urban': 1, 'Semiurban': 2})


# 4. XUẤT FILE BÀN GIAO
# Giữ nguyên thang đo gốc của các con số tiền tệ, không dùng StandardScaler
df_clean.to_csv('clean_loan_data.csv', index=False)

print("=> HOÀN TẤT 100%! Đã giữ lại cột 'Property_Area' và chuyển hóa thành số.")
print("=> File 'clean_loan_data.csv' đã sẵn sàng bàn giao!")

BƯỚC 5: PYTHON THỰC THI LÀM SẠCH VÀ SỐ HÓA
=> HOÀN TẤT 100%! Đã giữ lại cột 'Property_Area' và chuyển hóa thành số.
=> File 'clean_loan_data.csv' đã sẵn sàng bàn giao!
